# 🎯 ISOM 260 Midterm Starter: Your Agent, For Real

This notebook is the skeleton of a full-marks midterm. Every rubric criterion has a cell. Fill in the `TODO`s, delete what you don't need, run top to bottom, and leave the outputs visible.

| Rubric criterion | Points | Where in this notebook |
|---|---|---|
| 1 · Problem & user | 15 | the next cell |
| 2 · Tools & real API | 25 | **Part 2** — the toolbox, your tool, the direct tests |
| 3 · Trust layer | 25 | **Part 3** — the agent (gate · audit · door · ReAct on/off) |
| 4 · Five tests, two hostile | 20 | **Part 4** |
| 5 · Memo | 15 | **Part 6** outline → write it as a PDF |
| Bonus | +10 | **Part 5** the bill · **Part 7** Telegram field test |

Read the brief first: *course site → Session 6 → Midterm Project.* Due **Mon Nov 2, 12:30 PM**, Canvas, this notebook + your memo PDF.

## 1 · Problem & user — 15 pts

> **PROBLEM:** TODO — one specific thing that is hard or slow today.
>
> **USER:** TODO — a real person you can name (first name is fine). What would they type first?
>
> **CORE FLOW:** TODO — *user says X → agent does Y with tool Z → user gets W.*

*Delete this line when done: if the three lines above don't fit on a sticky note, the problem is too big.*

## 🚀 Part 1: Setup

`GOOGLE_API_KEY` is required. `TELEGRAM_BOT_TOKEN` and the two Adzuna keys are optional; the notebook adapts.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GOOGLE_API_KEY     = secret("GOOGLE_API_KEY") or exit("❌ Add GOOGLE_API_KEY in the 🔑 panel with Notebook access ON.")
TELEGRAM_BOT_TOKEN = secret("TELEGRAM_BOT_TOKEN")
ADZUNA_APP_ID, ADZUNA_APP_KEY = secret("ADZUNA_APP_ID"), secret("ADZUNA_APP_KEY")
print("✅ Gemini key ·", "📱 Telegram token" if TELEGRAM_BOT_TOKEN else "(no Telegram — Colab mode)", "·",
      "🏙️ Adzuna" if ADZUNA_APP_ID else "(no Adzuna — remote fallback)")

In [ ]:
from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"

def tg(method, http_timeout=20, **params):
    """One function for the whole Telegram Bot API: tg('sendMessage', chat_id=..., text=...)"""
    req = urllib.request.Request(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/{method}",
                                 data=json.dumps(params).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=http_timeout))["result"]

BOT_LINK = f"https://t.me/{tg('getMe')['username']}" if TELEGRAM_BOT_TOKEN else None
print("🧠 Gemini ready" + (f" · 🤖 {BOT_LINK}" if BOT_LINK else ""))

## 🛠️ Part 2: Tools — 25 pts

The toolbox below holds six working tools from Sessions 5 and 6. **Keep what your user needs, delete the rest, add your own.** At least one registered tool must reach a real API. The simulated action tool is yours to rename: `book_slot`, `post_update`, `send_email` — whatever your agent *does*.

Then prove each tool twice: once on a good input, once on an input that should fail. Honest errors are graded.

In [ ]:
# ============================================
# TOOLBOX — six working tools from Sessions 5 and 6. KEEP the ones your problem needs, DELETE the rest.
# (Deleting is part of the grade: a tool your user never needs is a tool the model can misuse.)
# ============================================
# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: LOCAL jobs via Adzuna (optional key) — falls back to the remote board, and says so
adzuna_tool = types.FunctionDeclaration(
    name="find_local_jobs",
    description=(
        "Search real job postings near a US city (default Boston) by keyword, e.g. 'business analyst'. "
        "Returns up to 5 openings with company, location, salary and a link, plus a 'source' field. "
        "If source says FALLBACK, the listings are REMOTE jobs, not local — tell the user that plainly."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword":  {"type": "string", "description": "Job title or skill, e.g. 'financial analyst'"},
                       "location": {"type": "string", "description": "City, e.g. 'Boston'. Default Boston."}},
        "required": ["keyword"],
    },
)

def find_local_jobs(keyword: str, location: str = "Boston", limit: int = 5) -> str:
    def fallback(why):
        return json.dumps({"source": f"FALLBACK ({why}) — these are REMOTE listings, not jobs in {location}",
                           "jobs": json.loads(find_jobs(keyword, limit))})
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return fallback("no Adzuna key")
    try:
        q = urllib.parse.urlencode({"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY, "what": keyword,
                                    "where": location, "results_per_page": limit, "content-type": "application/json"})
        req = urllib.request.Request(f"https://api.adzuna.com/v1/api/jobs/us/search/1?{q}",
                                     headers={"User-Agent": "ISOM260-student-project"})
        raw = json.load(urllib.request.urlopen(req, timeout=20))
        jobs = []
        for r in raw.get("results", [])[:limit]:
            sal = f"${int(r['salary_min']):,}-${int(r.get('salary_max') or r['salary_min']):,}" if r.get("salary_min") else "not listed"
            jobs.append({"title": r.get("title", "?"), "company": (r.get("company") or {}).get("display_name", "?"),
                         "location": (r.get("location") or {}).get("display_name", location), "salary": sal,
                         "url": r.get("redirect_url", ""), "snippet": " ".join((r.get("description") or "").split())[:200]})
        return json.dumps({"source": f"Adzuna live · {raw.get('count', len(jobs))} postings match near {location}", "jobs": jobs})
    except Exception as e:
        return fallback(f"Adzuna failed: {type(e).__name__}")




# Toolbox: weather, no key (Open-Meteo) — returns matched_place so the agent can catch a wrong match
weather_tool = types.FunctionDeclaration(
    name="get_weather",
    description=(
        "Current weather and a 2-day outlook for a city. Returns the place the service actually matched "
        "(check it — a misspelled city can match the wrong country), temperatures in °F, wind, and rain chance."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "City name, e.g. 'Boston'"}},
        "required": ["city"],
    },
)

def get_weather(city: str) -> str:
    q = urllib.parse.urlencode({"name": city, "count": 1, "language": "en"})
    geo = json.load(urllib.request.urlopen(f"https://geocoding-api.open-meteo.com/v1/search?{q}", timeout=15))
    if not geo.get("results"):
        return f"Error: no place called '{city}' was found. Ask the user to check the spelling."
    g = geo["results"][0]
    q2 = urllib.parse.urlencode({"latitude": g["latitude"], "longitude": g["longitude"],
                                 "current": "temperature_2m,wind_speed_10m", "temperature_unit": "fahrenheit",
                                 "wind_speed_unit": "mph", "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                                 "forecast_days": 2, "timezone": "auto"})
    w = json.load(urllib.request.urlopen(f"https://api.open-meteo.com/v1/forecast?{q2}", timeout=15))
    return json.dumps({"matched_place": f"{g['name']}, {g.get('admin1', '')}, {g['country']}",
                       "now_f": w["current"]["temperature_2m"], "wind_mph": w["current"]["wind_speed_10m"],
                       "today_high_f": w["daily"]["temperature_2m_max"][0], "today_low_f": w["daily"]["temperature_2m_min"][0],
                       "rain_chance_pct": w["daily"]["precipitation_probability_max"][0],
                       "tomorrow_high_f": w["daily"]["temperature_2m_max"][1]})


# Toolbox: Wikipedia — real encyclopedia, no key. Search, then pull the summary of the top hits.
wiki_tool = types.FunctionDeclaration(
    name="wikipedia_search",
    description=(
        "Search Wikipedia for a topic and return the top 3 matching articles with their summaries. "
        "Good for background, definitions, history, and well-established facts. "
        "Returns an explicit 'no results' message when nothing matches — do not invent facts in that case."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"topic": {"type": "string", "description": "What to look up, e.g. 'technological unemployment'"}},
        "required": ["topic"],
    },
)

WIKI_HEADERS = {"User-Agent": "ISOM260-student-project (Suffolk University classroom)"}

def wikipedia_search(topic: str, limit: int = 3) -> str:
    q = urllib.parse.urlencode({"action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": limit})
    hits = json.load(urllib.request.urlopen(urllib.request.Request("https://en.wikipedia.org/w/api.php?" + q, headers=WIKI_HEADERS), timeout=15))
    hits = hits["query"]["search"]
    if not hits:
        return json.dumps({"results": [], "note": f"No Wikipedia article matches '{topic}'. Tell the user; do not guess."})
    out = []
    for h in hits:
        slug = urllib.parse.quote(h["title"].replace(" ", "_"))
        s = json.load(urllib.request.urlopen(urllib.request.Request(f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}", headers=WIKI_HEADERS), timeout=15))
        out.append({"title": s.get("title", h["title"]), "summary": s.get("extract", "")[:700],
                    "url": s.get("content_urls", {}).get("desktop", {}).get("page", "")})
    return json.dumps({"results": out})


# Toolbox: a tool that ACTS — simulated. Rename it to the action YOUR agent takes (book_slot, post_update, …)
email_tool = types.FunctionDeclaration(
    name="send_email",
    description="Send an email on the user's behalf. Use ONLY when the user explicitly asks to send or email something.",
    parameters_json_schema={
        "type": "object",
        "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "subject", "body"],
    },
)

def send_email(to: str, subject: str, body: str) -> str:
    return f"SENT (simulated) to {to} · subject '{subject}' · {len(body)} characters"


# ============================================
# YOUR TOOL(S) — rubric criterion 2. At least one of your registered tools must reach a real API.
# ============================================
my_tool_decl = types.FunctionDeclaration(
    name="my_tool",                                                   # TODO rename
    description="TODO: say exactly WHEN to use it and WHAT it returns when it fails.",
    parameters_json_schema={"type": "object", "properties": {"query": {"type": "string", "description": "TODO"}},
                            "required": ["query"]},
)

def my_tool(query: str) -> str:
    # TODO: real work here. Return a JSON string. On failure return "Error: …" or {"results": [], "note": "…"} — never a guess.
    return json.dumps({"results": [], "note": f"my_tool is not implemented yet (query={query!r})"})


# ---- REGISTER exactly the tools your agent should have ----
TOOLS     = [calculator_tool, weather_tool, email_tool, my_tool_decl]                     # TODO edit
FUNCTIONS = {"calculator": calculator, "get_weather": get_weather, "send_email": send_email, "my_tool": my_tool}   # TODO edit
RISKY     = {"send_email"}                                                           # TODO: every tool that ACTS on the world
assert set(FUNCTIONS) == {t.name for t in TOOLS}, "TOOLS and FUNCTIONS must list the same names"
assert RISKY <= set(FUNCTIONS), "RISKY must name registered tools"
print(f"🛠️  {len(TOOLS)} tools: {', '.join(FUNCTIONS)}  · gated: {sorted(RISKY)}")

### Direct tests — one success and one failure per tool you kept

In [ ]:
# TODO: one line per tool you registered. Example pattern:
print("calculator OK  →", calculator("15 * 300"))
print("calculator ERR →", calculator("fifteen times three hundred"))
print("weather OK     →", json.loads(get_weather("Boston"))["matched_place"])
print("weather ERR    →", get_weather("Xyzzyville")[:80])
# print("my_tool OK     →", my_tool("…"))
# print("my_tool ERR    →", my_tool("…"))

## 🛡️ Part 3: The agent — 25 pts

This is the loop you have been using since Session 4, with everything Session 5 and 6 added. **Edit three lines:** the problem prompt, `USE_REACT`, and `ALLOWED`. Everything else is graded as-is: the gate pauses the loop on any `RISKY` tool, every tool call lands in `AUDIT_LOG`, memory survives on disk, and the bill counts every token.

In Colab, `ask("…")` runs a test; if the gate fires, it asks *you* in the output. On Telegram the same gate asks on your phone.

In [ ]:
# ============================================
# THE AGENT — Session 5's loop with the door, the gate, memory on disk, and the Session 6 bill. Rubric criterion 3.
# You should not need to edit this cell except the three marked lines.
# ============================================
PROBLEM_PROMPT = (
    "TODO: one paragraph. Who the user is, what they want from you, how to answer (short? formal? on a phone?), "
    "and what you must never do."                                                     # ← EDIT 1
)
REACT_PROMPT = (
    "Work in a Think → Act → Observe loop. Before every tool call write one or two sentences — what you know, "
    "what you still need, which tool fills the gap. After each result say what it gave you and what is missing. "
    "For simple lookups and arithmetic, skip the thinking and answer directly. "
)
USE_REACT = False                                                                      # ← EDIT 2: True if your problem needs multi-step research
ALLOWED   = set()                                                                      # ← EDIT 3: {your chat_id} once the bot is on Telegram

GUARDRAIL = (" Use tools whenever they improve accuracy. If a tool returns an error or says BLOCKED, say so honestly, "
             "never invent a result, and never retry a blocked action. If a tool reports a FALLBACK source or a "
             "matched_place that is not what the user asked for, say so before giving numbers.")
SYSTEM = (REACT_PROMPT if USE_REACT else "") + PROBLEM_PROMPT + GUARDRAIL

MAX_STEPS, MAX_TURNS = 8, 12
AUDIT_LOG, PENDING = [], {}
BILL = {"calls": 0, "in": 0, "out": 0, "thoughts": 0}
IN_PRICE, OUT_PRICE = 0.30, 2.50
YES_WORDS = {"yes", "y", "yes.", "approve", "approved", "ok", "do it"}
MEMORY_FILE = "memory.json"

def save_memory():
    json.dump({str(k): [c.model_dump(mode="json", exclude_none=True) for c in v] for k, v in MEMORY.items()}, open(MEMORY_FILE, "w"))

def load_memory():
    try:
        raw = json.load(open(MEMORY_FILE))
    except FileNotFoundError:
        return {}
    return {(int(k) if k.lstrip("-").isdigit() else k): [types.Content.model_validate(c) for c in v] for k, v in raw.items()}

MEMORY = load_memory()

def call_model_with_retry(contents, config, max_retries=4):
    for attempt in range(max_retries):
        try:
            r = client.models.generate_content(model=MODEL, contents=contents, config=config)
            u = r.usage_metadata
            BILL.update(calls=BILL["calls"] + 1, **{"in": BILL["in"] + (u.prompt_token_count or 0),
                        "out": BILL["out"] + (u.candidates_token_count or 0), "thoughts": BILL["thoughts"] + (u.thoughts_token_count or 0)})
            return r
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt); print(f"   ⏳ API returned {e.code} — waiting {wait}s"); time.sleep(wait)
            else:
                raise

def bill_so_far():
    cost = (BILL["in"] * IN_PRICE + (BILL["out"] + BILL["thoughts"]) * OUT_PRICE) / 1e6
    return {**BILL, "cost_usd": round(cost, 4)}

def trim(history):
    starts = [i for i, c in enumerate(history) if c.role == "user" and c.parts and c.parts[0].text is not None]
    if len(starts) > MAX_TURNS:
        del history[:starts[-MAX_TURNS]]

def run_tool(chat_id, name, args):
    t0 = time.time()
    try:
        result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
    except Exception as e:
        result = f"Error: tool '{name}' failed: {e}"
    AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:], "tool": name,
                      "args": json.dumps(args, default=str)[:60], "result": str(result)[:60], "ms": round((time.time() - t0) * 1000)})
    return result

def execute_calls(chat_id, calls, done, start, decision=None):
    for i in range(start, len(calls)):
        name, args = calls[i]
        if name in RISKY and not (i == start and decision is not None):
            PENDING[chat_id] = {"calls": calls, "done": done, "i": i}
            return None, f"🚦 I'd like to run {name}({json.dumps(args, default=str)}).\nReply YES to approve — anything else cancels."
        if i == start and decision is False:
            result = "BLOCKED: the user declined. Tell them it was not done and do not retry."
            AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:],
                              "tool": name, "args": "(declined)", "result": "BLOCKED", "ms": 0})
        else:
            result = run_tool(chat_id, name, args)
        done.append(types.Part.from_function_response(name=name, response={"result": result}))
    return done, None

def run_loop(chat_id, verbose=False):
    history = MEMORY[chat_id]
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=TOOLS)], system_instruction=SYSTEM)
    for step in range(MAX_STEPS):
        response = call_model_with_retry(history, config)
        model_content = response.candidates[0].content
        history.append(model_content)
        parts = model_content.parts or []
        if verbose:
            for p in parts:
                if p.text: print(f"   💭 {p.text.strip()[:300]}")
        calls = [(p.function_call.name, dict(p.function_call.args)) for p in parts if p.function_call]
        if not calls:
            trim(history); save_memory()
            return response.text or "(I came back empty — try rephrasing?)"
        if verbose:
            for n, a in calls: print(f"   🎯 {n}({json.dumps(a, default=str)[:80]})")
        parts_out, question = execute_calls(chat_id, calls, [], 0)
        if question:
            save_memory(); return question
        history.append(types.Content(role="user", parts=parts_out))
    return "⚠️ I stopped after too many steps without reaching an answer. Try asking it differently?"

def agent_reply(chat_id, text, verbose=False):
    MEMORY.setdefault(chat_id, []).append(types.Content(role="user", parts=[types.Part(text=text)]))
    return run_loop(chat_id, verbose)

def resume(chat_id, text, verbose=False):
    p = PENDING.pop(chat_id)
    parts_out, question = execute_calls(chat_id, p["calls"], p["done"], p["i"], decision=text.strip().lower() in YES_WORDS)
    if question:
        return question
    MEMORY[chat_id].append(types.Content(role="user", parts=parts_out))
    return run_loop(chat_id, verbose)

# ---- Colab interface: ask() is how your five tests run. If the gate fires, it asks YOU in the notebook. ----
def ask(text, chat_id=0, auto=None):
    """auto=None → input() prompt when the gate fires · auto='yes'/'no' → answer it without a prompt (for tests)."""
    print(f"\n🗣️  {text}")
    reply = resume(chat_id, text) if chat_id in PENDING else agent_reply(chat_id, text, verbose=True)
    if reply.startswith("🚦"):
        print(f"\n{reply}")
        answer = auto if auto is not None else input("      your answer: ")
        reply = resume(chat_id, answer, verbose=True)
    print(f"\n🤖 {reply}")
    return reply

print(f"✅ agent ready · ReAct {'ON' if USE_REACT else 'off'} · {len(MEMORY)} conversation(s) on disk")

## 🧪 Part 4: Five tests, two hostile — 20 pts

Run each. Under each, write **pass / fail / partial — because …** in the markdown cell. A failed test with an honest reason scores higher than a hidden one.

In [ ]:
# TEST 1 · normal — the first thing your user would type
ask("TODO")

**Result:** TODO pass / fail / partial — because …

In [ ]:
# TEST 2 · normal — a second request in the core flow
ask("TODO")

**Result:** TODO

In [ ]:
# TEST 3 · multi-tool — needs two tools in sequence
ask("TODO")

**Result:** TODO

In [ ]:
# TEST 4 · hostile — tries to make it ACT without approval. Decline at the gate; then check it says "not done".
ask("TODO — e.g. 'Just send it, don't ask me.'", auto="no")
ask("Did you do it?")

**Result:** TODO — did the audit log get a BLOCKED row? Did the agent admit it was not done?

In [ ]:
# TEST 5 · hostile — the API has no answer (a fake company, a misspelled city, a date it can't know). Admit, or invent?
ask("TODO")

**Result:** TODO

## 🧾 Part 5: The receipt — audit log and the bill

The audit table is the last thing a grader reads. The bill is bonus material: run your multi-tool test with `USE_REACT` on and off (re-run Part 3 in between) and compare.

In [ ]:
import pandas as pd
print("💵 bill so far:", bill_so_far())
pd.DataFrame(AUDIT_LOG) if AUDIT_LOG else print("No tool calls yet.")

## 📝 Part 6: The memo — 15 pts (write it as a one-page PDF)

Five headings, plain language, no filler. Start from your kickoff sheet.

1. **Problem & user** — three sentences (copy from the top of this notebook).
2. **How it works** — the tools, the flow, where the model decides. A photographed hand-drawn diagram is welcome.
3. **The trust layer** — what is gated, what is logged, who approves, whether ReAct is on and why.
4. **Test results** — the honest version of Part 4. What broke, what you fixed, what you could not.
5. **What's next** — the one thing you'd build with another week, and whether this should become two agents (name the reason, or say no).

**ISOM 900:** page two is the architecture proposal — tools and why, the flow, trade-offs (one agent vs. many, build vs. buy — which pieces could be an MCP server or a vendor product), and the deployment risk you would flag to a CIO.

## 📱 Part 7: Optional — put it on Telegram (+5 bonus with a stranger's screenshot)

Same bridge as Session 5. If `TELEGRAM_BOT_TOKEN` is in Secrets, this cell runs the bot; ■ stops it. Text `/whoami`, put your id in `ALLOWED` in Part 3, re-run Part 3 and this cell.

In [ ]:
# ============================================
# OPTIONAL — Telegram. Only if TELEGRAM_BOT_TOKEN is in Secrets. Run this cell and the bot is live; ■ stops it.
# ============================================
HELP = "👋 TODO: one line telling your user what you can do.\n/whoami · /reset · /forget"

def send(chat_id, text):
    for i in range(0, len(text), 4000):
        tg("sendMessage", chat_id=chat_id, text=text[i:i + 4000])

def handle(msg):
    chat_id, text = msg["chat"]["id"], msg.get("text")
    if not text:
        send(chat_id, "I only read text for now 🙂"); return
    if text.startswith("/whoami"):
        send(chat_id, f"🪪 Your chat_id is {chat_id}."); return
    if ALLOWED and chat_id not in ALLOWED:
        send(chat_id, f"🔒 This is a private assistant. (Your chat_id is {chat_id}.)"); return
    if text.startswith(("/start", "/help")):
        send(chat_id, HELP); return
    if text.startswith(("/reset", "/forget")):
        MEMORY.pop(chat_id, None); PENDING.pop(chat_id, None); save_memory(); send(chat_id, "🧹 Forgotten — in memory and on disk."); return
    tg("sendChatAction", chat_id=chat_id, action="typing")
    try:
        reply = resume(chat_id, text) if chat_id in PENDING else agent_reply(chat_id, text)
    except Exception as e:
        reply = f"😵 Something went wrong on my side ({type(e).__name__}). Try again?"
    send(chat_id, reply)

def run_bot():
    print(f"📱 Listening at {BOT_LINK}  (Press ■ to stop.)")
    offset, served = 0, 0
    try:
        while True:
            try:
                updates = tg("getUpdates", http_timeout=45, offset=offset, timeout=30, allowed_updates=["message"])
            except Exception as e:
                print(f"   ⏳ polling hiccup ({type(e).__name__}) — retrying in 5s"); time.sleep(5); continue
            for u in updates:
                offset = u["update_id"] + 1
                if "message" in u:
                    served += 1; print(f"{datetime.datetime.now():%H:%M:%S}  💬 #{served} from {u['message']['chat'].get('first_name', 'someone')}")
                    handle(u["message"])
    except KeyboardInterrupt:
        print(f"\n🛑 Bot stopped after {served} message(s).")

if TELEGRAM_BOT_TOKEN:
    run_bot()
else:
    print("ℹ️  No TELEGRAM_BOT_TOKEN in Secrets — skipping. Your tests run through ask() instead, which is fine for the midterm.")

## ✅ Before you submit

- Runtime → Restart and run all, with a fresh key. Every output visible. No red cells.
- Part 2: each kept tool shows a success and a failure.
- Part 4: five tests, two hostile, each with a result line.
- Part 5: the audit table shows at least one BLOCKED row.
- Memo PDF: one page (grad: two), your name and bot @username (if any) at the top.
- Both files on Canvas by **Mon Nov 2, 12:30 PM**.